# Module 3 — Cyclone Risk Model for Indian Coastal Cities

This notebook builds a cyclone-risk classifier using historical wind and precipitation data from the **Open-Meteo Archive API** (2015-2023) for 10 Indian coastal cities.

**Pipeline:**
1. Fetch daily wind-speed, wind-gust, and rainfall data
2. Aggregate to yearly features (extreme-wind-day counts, annual maxima)
3. Label years as cyclone-affected based on sustained high-wind days
4. Train an XGBoost classifier with class-imbalance weighting
5. Compute risk scores and categories
6. Save outputs (CSV + model pickle)

In [ ]:
# ── Cell 1: Fetch wind / precipitation data for 10 coastal cities ──────────

import requests, time, warnings
import pandas as pd
import numpy as np

warnings.filterwarnings('ignore')

CITIES = [
    {"name": "Mumbai",         "lat": 19.08, "lon": 72.88},
    {"name": "Chennai",        "lat": 13.08, "lon": 80.27},
    {"name": "Kolkata",        "lat": 22.57, "lon": 88.36},
    {"name": "Bhubaneswar",    "lat": 20.30, "lon": 85.82},
    {"name": "Visakhapatnam",  "lat": 17.69, "lon": 83.22},
    {"name": "Surat",          "lat": 21.17, "lon": 72.83},
    {"name": "Kochi",          "lat":  9.93, "lon": 76.27},
    {"name": "Mangaluru",      "lat": 12.91, "lon": 74.86},
    {"name": "Puducherry",     "lat": 11.94, "lon": 79.81},
    {"name": "Goa",            "lat": 15.49, "lon": 73.83},
]

API_URL = "https://archive-api.open-meteo.com/v1/archive"
START_DATE = "2015-01-01"
END_DATE   = "2023-12-31"

frames = []
for city in CITIES:
    print(f"Fetching data for {city['name']}...")
    params = {
        "latitude":   city["lat"],
        "longitude":  city["lon"],
        "start_date": START_DATE,
        "end_date":   END_DATE,
        "daily":      "wind_speed_10m_max,wind_gusts_10m_max,precipitation_sum",
        "timezone":   "Asia/Kolkata",
    }
    resp = requests.get(API_URL, params=params, timeout=30)
    resp.raise_for_status()
    data = resp.json()["daily"]

    tmp = pd.DataFrame({
        "date":               pd.to_datetime(data["time"]),
        "wind_speed_max_kmh": data["wind_speed_10m_max"],
        "wind_gust_max_kmh":  data["wind_gusts_10m_max"],
        "rainfall_mm":        data["precipitation_sum"],
        "city":               city["name"],
        "lat":                city["lat"],
        "lon":                city["lon"],
    })
    frames.append(tmp)
    time.sleep(0.3)

df_all = pd.concat(frames, ignore_index=True)
df_all.fillna(0, inplace=True)

print(f"\nCombined shape: {df_all.shape}")
df_all.head()

In [ ]:
# ── Cell 2: Feature engineering — yearly aggregation ──────────────────────

df_all["year"] = df_all["date"].dt.year

yearly = df_all.groupby(["city", "year", "lat", "lon"]).agg(
    days_wind_above_90kmh  = ("wind_speed_max_kmh", lambda s: (s > 90).sum()),
    days_wind_above_120kmh = ("wind_speed_max_kmh", lambda s: (s > 120).sum()),
    max_windspeed_year     = ("wind_speed_max_kmh", "max"),
    max_windgust_year      = ("wind_gust_max_kmh",  "max"),
).reset_index()

print(f"Yearly aggregation shape: {yearly.shape}")
yearly.head(10)

In [ ]:
# ── Cell 3: Cyclone label ────────────────────────────────────────────────
# A city-year is labelled 1 (cyclone-affected) when it has >2 days with
# max wind speed above 90 km/h.  The dataset is expected to be imbalanced.

yearly["cyclone_label"] = (yearly["days_wind_above_90kmh"] > 2).astype(int)

print("Label distribution:")
print(yearly["cyclone_label"].value_counts())
print(f"\nPositive rate: {yearly['cyclone_label'].mean():.2%}")

In [ ]:
# ── Cell 4: Train XGBoost classifier ─────────────────────────────────────

import os, joblib
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
)
import matplotlib.pyplot as plt

FEATURES = ["max_windspeed_year", "max_windgust_year", "days_wind_above_120kmh"]
TARGET   = "cyclone_label"

# Train / test split by year
train = yearly[yearly["year"] <= 2021]
test  = yearly[yearly["year"] >= 2022]

X_train, y_train = train[FEATURES], train[TARGET]
X_test,  y_test  = test[FEATURES],  test[TARGET]

# Handle class imbalance
scale_pos_weight = len(y_train[y_train == 0]) / max(len(y_train[y_train == 1]), 1)
print(f"scale_pos_weight = {scale_pos_weight:.2f}")

model = XGBClassifier(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=42,
)
model.fit(X_train, y_train)

y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

print(f"\nAccuracy : {accuracy_score(y_test, y_pred):.4f}")
print(f"Precision: {precision_score(y_test, y_pred, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred, zero_division=0):.4f}")
print(f"F1       : {f1_score(y_test, y_pred, zero_division=0):.4f}")

try:
    auc = roc_auc_score(y_test, y_proba[:, 1])
    print(f"AUC-ROC  : {auc:.4f}")
except ValueError as e:
    print(f"AUC-ROC  : N/A ({e})")

# Feature importance plot
os.makedirs("../data/outputs", exist_ok=True)

fig, ax = plt.subplots(figsize=(8, 4))
importances = model.feature_importances_
ax.barh(FEATURES, importances, color="#2196F3")
ax.set_xlabel("Importance")
ax.set_title("Cyclone Model — Feature Importance")
fig.tight_layout()
fig.savefig("../data/outputs/cyclone_feature_importance.png", dpi=150)
plt.show()
print("Saved: ../data/outputs/cyclone_feature_importance.png")

In [ ]:
# ── Cell 5: Risk scores & categories ─────────────────────────────────────

# Score the FULL dataset (all city-years)
all_proba = model.predict_proba(yearly[FEATURES])[:, 1]
yearly["cyclone_prob"]       = all_proba
yearly["cyclone_risk_score"] = (all_proba * 100).round(2)

def risk_category(score):
    if score <= 25:
        return "LOW"
    elif score <= 50:
        return "MEDIUM"
    elif score <= 75:
        return "HIGH"
    else:
        return "VERY HIGH"

yearly["risk_category"] = yearly["cyclone_risk_score"].apply(risk_category)

risk_scores = yearly[[
    "city", "year", "lat", "lon",
    "cyclone_label", "cyclone_prob", "cyclone_risk_score", "risk_category",
]].copy()

print(risk_scores.to_string(index=False))

In [ ]:
# ── Cell 6: Save outputs ─────────────────────────────────────────────────

risk_scores.to_csv("../data/outputs/cyclone_risk_scores.csv", index=False)
print("Saved: ../data/outputs/cyclone_risk_scores.csv")

joblib.dump(model, "../data/outputs/cyclone_model.pkl")
print("Saved: ../data/outputs/cyclone_model.pkl")

print("\n✅ Module 3 — Cyclone Risk complete.")